# Exercise 7: GQA Attention（无 KV cache）

**Goal:** 实现 Grouped Query Attention

GQA = Q 有完整的 `num_heads` 个头，但 KV 只有 `num_kv_heads` 个头（更少）。每个 KV 头服务多个 Q 头。

**Qwen3 示例:** `num_heads=16, num_kv_heads=8` → 每个 KV 头服务 2 个 Q 头

**实现步骤：**
```
1. x → q_proj, k_proj, v_proj
   q: (B, T, num_heads * head_dim)
   k: (B, T, num_kv_heads * head_dim)
   v: (B, T, num_kv_heads * head_dim)

2. reshape → (B, T, n_heads, head_dim)，transpose → (B, n_heads, T, head_dim)

3. 对 q, k 施加 RoPE

4. 把 k, v repeat 到 num_heads 个头
   k: (B, num_kv_heads, T, head_dim) → (B, num_heads, T, head_dim)

5. scaled dot-product attention:
   scores = (q @ k.transpose(-2,-1)) / sqrt(head_dim)
   weights = softmax(scores, dim=-1)
   out = weights @ v

6. reshape → (B, T, hidden_size)，经过 o_proj
```

**写之前先在脑子里回答：**
- `repeat_interleave` 和 `repeat` 有什么区别？
- 为什么除以 `sqrt(head_dim)`？
- causal mask 应该加在哪一步？（本练习先跳过）

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# 把上一个练习的 RoPE 函数粘贴到这里（或者 import）
# from exercise_02_rope import precompute_freqs, apply_rope

In [ ]:
# 如果还没写 RoPE，先用这个 dummy 版本跑通 attention，之后再替换
def precompute_freqs(head_dim, max_seq_len, base=10000.0):
    cos = torch.ones(max_seq_len, head_dim)
    sin = torch.zeros(max_seq_len, head_dim)
    return cos, sin

def apply_rope(q, k, cos, sin):
    return q, k  # 占位，不做旋转

In [ ]:
class MyGQAAttention(nn.Module):
    def __init__(self, hidden_size: int, num_heads: int, num_kv_heads: int):
        super().__init__()
        assert hidden_size % num_heads == 0
        self.num_heads    = num_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim     = hidden_size // num_heads
        self.groups       = num_heads // num_kv_heads
        # YOUR CODE HERE: q_proj, k_proj, v_proj, o_proj

    def forward(
        self,
        x: torch.Tensor,    # (B, T, hidden_size)
        cos: torch.Tensor,  # (T, head_dim)
        sin: torch.Tensor,  # (T, head_dim)
    ) -> torch.Tensor:
        # YOUR CODE HERE
        pass

In [ ]:
# Verification
torch.manual_seed(0)
B, T = 2, 6
hidden, n_heads, n_kv = 64, 4, 2
head_dim = hidden // n_heads

attn = MyGQAAttention(hidden, n_heads, n_kv)
x    = torch.randn(B, T, hidden)
cos, sin = precompute_freqs(head_dim, T)

out = attn(x, cos, sin)
print("input  shape:", x.shape)
print("output shape:", out.shape)
assert out.shape == (B, T, hidden), f"Expected ({B}, {T}, {hidden}), got {out.shape}"
print("PASS")